<a href="https://colab.research.google.com/github/kimdesok/ViT-backbone-MIL-on-CAMELYON16/blob/main/MIL_ViT_CAME_Inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import rebel # RBLN Compiler OMG!

import numpy as np
import pandas as pd

import io
import gc

import tensorflow as tf
tf.get_logger().setLevel("WARNING")
#tf.config.run_functions_eagerly(True)

import keras
import cv2
import keras_hub

import openslide
import random

from keras import layers
from keras import ops
from tqdm import tqdm
from matplotlib import pyplot as plt

plt.style.use("ggplot")
print("TF version:", tf.__version__)
print("Keras version:", keras.__version__)

import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"   # only warnings & errors

# Force TensorFlow to see all GPUs
os.environ["CUDA_VISIBLE_DEVICES"] = "0, 1"
# suppressing useless backend logs
os.environ['TF_XLA_FLAGS'] = '--tf_xla_enable_xla_devices=false'

import warnings
warnings.filterwarnings("ignore")

2025-09-15 10:47:32.854794: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
/home/elicer/tf219_py310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


TF version: 2.19.0
Keras version: 3.11.3


## Setting parameters

In [ ]:
##### Models
#model_name = "CNN_back_MIL"
#model_name = "ResNet50_back_MIL"
#model_name = "VGG19_back_MIL"
#model_name = "ViT_back_MIL"
model_name = "MIL_CNN_model"

##### Datasets
data_name = "came16"
mag_label = "10x"
slide_dir = "/home/elicer/data/camelyon16/training"
input_path = "/home/elicer/data/camelyon16/tfrecord_10x"  # "/home/elicer/data/camelyon16/tfrecord_2.5x"

# Paths to your TFRecord files (generated by convert_slides_to_tfrecord)
test_tfrecords_path  = [os.path.join(input_path, "test.tfrecord")]
print(test_tfrecords_path)

# a new way to split the train_tfrecords into train and test dataset file list
data_used = test_tfrecords_path[0].split("/")[-1].split("_")[0]
if(data_used == "test.tfrecord"):
    data_used = ""
print("Data label", data_used)

# Paths for output performance, model, & training graphs
csv_file_path = "./perform_data/hyperparameter_search_camelyon16.csv"
#model_pre_path = f"./models/0.9446_VGG19_back_MIL_lr0.0001_ppg32_bs32_nb128_nr0.25.keras"
#model_pre_path = f"./models/0.9692_MIL_CNN_model_vbc500_bc1000_lr0.001_b4.keras"
#model_pre_path = f"./models/0.9366_ResNet50_backbone_MIL_lr0.0001_ppg32_bs32_nb128.keras"
#model_pre_path = f"./models/0.9394_Inception_back_MIL_lr0.0001_ppg32_bs32_nb128_nr0.25.keras
model_pre_path =f"./models/0.9831_Jk_ViT_back_MIL_came16_10x_lr1e-05_ppg70_bs8_nb128_nr0.95_model_vit_base_patch16_224_imagenet.keras"
print("Model path", model_pre_path)

# Define hyperparameter search space
learning_rates = [1.0e-05] # Learning rates to test  0.001,128,0.01,0.6,0.5
batch_sizes = [8] #16,32,64,128,256, # Batch sizes

# Hypers for MIL
bag_counts = [9000]  # 9000 for real training
val_bag_counts = [3000] # 3000 for real training
patches_per_bags = [70] # 16, 32, 64, 128
bag_sizes = [70] # [16, 32, 64, 128]
num_buckets = [128]
norm_ratios = [0.95] #0.99, 0.98, 0.97, 0.96

bag_counts_pairs = list(zip(bag_counts, val_bag_counts)) # No cross-mixing but pairing elements position-wise

# Initial hyperparameters for epoch 1 and print out
learning_rate = learning_rates[0]
batch_size = batch_sizes[0]

bag_count = bag_counts[0]
val_bag_count = val_bag_counts[0]
patches_per_bag = patches_per_bags[0]
bag_size = bag_sizes[0]
num_bucket = num_buckets[0]
norm_ratio = norm_ratios[0]

# Hyperparameters for training
epochs = 20 # 20 for real
drop_out = 0.5

n_early_stop = 5
n_reduce_lr = 3
n_factor = 0.6

monitor = "val_loss"  #"val_auc"
if(monitor == "val_loss"):
    mode= "min"
else:
    mode = "max"

# selecting the type of a pretrained ViT model
pretrained_vit_list=list(keras_hub.models.ViTBackbone.presets.keys())
#print(pretrained_vit_list)
pretrained_vit = pretrained_vit_list[0]
print(f"\nTaking {pretrained_vit} as the pretrained ViT model")

patch_size = 224
instance_shape = (patch_size, patch_size, 3)

# Workflow controls
quantize_model = False
inference_only = False
check_dataset = False  # inspect datasets' integrity such as record shapes etc
RECOVER = False   # recover datasets from the corrupted datasets
check_corruption = False # check if some records are corrupted
check_balance = False # check if normals and tumors well balanced as 1:1 mostly.

#Misc.
machine_name = "A100x2"
no_svs = 0  # Convert all svs files
verbose = 1
comments = f"test before full run, epochs={epochs}, drop_out={drop_out}, early stop={n_early_stop}, \
reduce_lr={n_reduce_lr}, factor={n_factor}, pretrained={pretrained_vit}, magnif={mag_label}, dataset={data_name}"


['/home/elicer/data/camelyon16/tfrecord_10x/test.tfrecord']
Data label 
Model path ./models/0.9831_Jk_ViT_back_MIL_came16_10x_lr1e-05_ppg70_bs8_nb128_nr0.95_model_vit_base_patch16_224_imagenet.keras

Taking vit_base_patch16_224_imagenet as the pretrained ViT model


In [ ]:
import psutil
from tensorflow.keras import mixed_precision

total_ramx = psutil.virtual_memory().total / (1024**3)  # Convert bytes to GB
print(f"Total RAM: {total_ramx:.2f} GB")

# floating point 16 operation
policy = mixed_precision.Policy('mixed_float16')
mixed_precision.set_global_policy(policy)

print(f"CUDA version: {tf.sysconfig.get_build_info()['cuda_version']}")
print(f"cuDNN version: {tf.sysconfig.get_build_info()['cudnn_version']}")

Total RAM: 72.00 GB
CUDA version: 12.5.1
cuDNN version: 9


In [ ]:
devices = tf.config.list_physical_devices()

# Prevents TensorFlow from allocating all GPU memory to one request.
if devices:
    for device in devices:
        if(device == "GPU"):
            tf.config.experimental.set_memory_growth(device, True)
    print("✅ Devices are available:", devices)
else:
    print("🚨 No Devices found! TensorFlow is running on CPU.")

✅ Devices are available: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]


## Build MIL bags with tf.data

In [ ]:
import math
import random

def inspect_random_bucket(parsed_ds,
                          num_buckets: int,
                          patches_to_show: int = 16,
                          max_slides_to_scan: int = 500,
                          max_cols: int = 4,
                          figsize: tuple = (8, 8)):
    """
    Randomly pick one slide’s bucket from parsed_ds and visualize its patches.

    parsed_ds: Dataset yielding (image: [H,W,3] float32 [0,1], label)
    num_buckets: number of hash buckets for slide IDs
    patches_to_show: number of patches to display
    max_slides_to_scan: how many *distinct* slide IDs to collect before choosing
    max_cols, figsize: controls the matplotlib grid
    """
    # 1) Collect unique slide IDs (bytes) up to `max_slides_to_scan`
    unique_sids = []
    for sid in parsed_ds.map(lambda sid, img, lbl: sid).as_numpy_iterator():
        if sid not in unique_sids:
            unique_sids.append(sid)
        if len(unique_sids) >= max_slides_to_scan:
            break

    if not unique_sids:
        raise ValueError("No slide IDs found in parsed_ds")

    # 2) Pick one at random
    chosen_sid = random.choice(unique_sids)
    chosen_sid_str = chosen_sid.decode()
    bucket_key = tf.strings.to_hash_bucket_fast(chosen_sid, num_buckets).numpy()

    # 3) Re-key the full parsed_ds so we can filter by bucket
    keyed = parsed_ds.map(
        lambda sid, img, lbl: (
            tf.strings.to_hash_bucket_fast(sid, num_buckets),
            sid, img, lbl
        ),
        num_parallel_calls=tf.data.AUTOTUNE
    )

    # 4) Filter exactly that bucket
    bucket_ds = keyed.filter(lambda k, sid, img, lbl: tf.equal(k, bucket_key))

    # 5) Pull out patches and metadata
    patches = []
    print(f"\n🔎 Inspecting bucket {bucket_key}  \n")
    for i, (k, img, lbl) in enumerate(bucket_ds.take(patches_to_show)):
        #sid_str = sid.numpy().decode()
        lbl_int = int(lbl.numpy())
        print(f"  Patch {i+1:02d}  label={lbl_int}")

        # convert [0,1] float → [0,255] uint8
        arr = (img.numpy() * 255).clip(0,255).astype(np.uint8)
        patches.append(arr)

    if not patches:
        print("⚠️ No patches found for that bucket.")
        return

    # 6) Plot them in an RGB grid (no colormap)
    cols = min(max_cols, len(patches))
    rows = math.ceil(len(patches) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = axes.flatten()
    for ax, patch in zip(axes, patches):
        ax.imshow(patch, interpolation="nearest")
        ax.axis("off")
    for ax in axes[len(patches):]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

def _parse_example(example_proto, patch_size=patch_size):
    """
    Parse a single TFRecord Example into (slide_id, image, label).
    """
    features = {
        'image_raw': tf.io.FixedLenFeature([], tf.string),
        'slide_id': tf.io.FixedLenFeature([], tf.string),
        'label': tf.io.FixedLenFeature([], tf.int64)
    }
    parsed = tf.io.parse_single_example(example_proto, features)
    image = tf.image.decode_png(parsed['image_raw'], channels=3)
    image = tf.image.convert_image_dtype(image, tf.float32)
    #image = tf.image.resize(image, [patch_size, patch_size])
    return parsed['slide_id'], image, parsed['label']

def augment_image(image, label):

    # Normalize to float32 in [0, 1]
    image = tf.cast(image, tf.float32) / 255.0

    # Geometric augmentations
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_flip_up_down(image)
    image = tf.image.rot90(image, k=tf.random.uniform(shape=[], minval=0, maxval=4, dtype=tf.int32))  # 0,90,180,270 rotation
    image = tf.image.resize_with_crop_or_pad(image, 256, 256)  # Zoom effect
    image = tf.image.random_crop(image, size=[224, 224, 3])

    # H&E-preserving color augmentations
    image = tf.image.random_brightness(image, max_delta=0.01)  # simulate scanner brightness
    image = tf.image.random_contrast(image, lower=0.99, upper=1.01)
    image = tf.image.random_saturation(image, lower=0.99, upper=1.01)
    image = tf.image.random_hue(image, max_delta=0.01)  # Slight hue shift, careful!

    # Optional: add mild noise or blur (simulating OOF)
    image = image + tf.random.normal(tf.shape(image), mean=0.0, stddev=0.01)

    # Clip and cast back
    image = tf.clip_by_value(image, 0.0, 1.0)
    image = tf.cast(image* 255.0, tf.uint8)

    return image, label

def create_bag_dataset(raw_ds, patches_per_bag=16, patch_size=patch_size, num_buckets=512, \
                       oversample_weight=(0.5, 0.5), training=False):
    """
    Returns a tf.data.Dataset yielding (bag_images, bag_label) per slide.
    bag_images: [patches_per_bag, patch_size, patch_size, 3]
    bag_label: max label among patches
    """
    def key_fn(key, image, label):
        #print("slide_id type:", type(key))
        return key

    def reduce_fn(key, ds):
        # ds yields (key, image, label) for one slide-bucket
        imgs = ds.map(lambda k, img, lbl: img, num_parallel_calls=tf.data.AUTOTUNE).batch(patches_per_bag, drop_remainder=True)
        labs = ds.map(lambda k, img, lbl: lbl, num_parallel_calls=tf.data.AUTOTUNE).batch(patches_per_bag, drop_remainder=True)
        bag_ds = tf.data.Dataset.zip((imgs, labs))
        return bag_ds.map(lambda imgs, labels: (imgs, tf.math.reduce_max(labels)), num_parallel_calls=tf.data.AUTOTUNE)

    #raw_ds = tf.data.TFRecordDataset(tfrecord_files, num_parallel_reads=4)

    parsed_ds = raw_ds.map(lambda x: _parse_example(x, patch_size), num_parallel_calls=tf.data.AUTOTUNE)

    if training:
        parsed_ds = parsed_ds.map(lambda sid, img, lbl: (sid, *augment_image(img, lbl)), num_parallel_calls=tf.data.AUTOTUNE)

    #for sid, img, lbl in parsed_ds.take(10):
    #    print(lbl.numpy())
    count_0, count_1 = 0, 0
    for _, _, lbl in parsed_ds.take(50000):
        if lbl.numpy() == 0:
            count_0 += 1
        else:
            count_1 += 1
    print("In Create_bag_dataset, Normal:", count_0, "Tumor:", count_1)


    # Map slide_id (string) to an integer bucket key
    keyed_ds = parsed_ds.map(
        lambda sid, img, lbl: (
            tf.strings.to_hash_bucket_fast(sid, num_buckets), img, lbl
        ),
        num_parallel_calls=tf.data.AUTOTUNE
    )

    bag_ds = keyed_ds.group_by_window(
        key_func=key_fn,
        reduce_func=reduce_fn,
        window_size=patches_per_bag * 10  # heuristic window size
    )

    #Balance the nos of tumor vs normal bags based on the oversample_weight
    tumor_ds  = bag_ds.filter(lambda imgs, lbl: tf.equal(lbl, 1))
    normal_ds = bag_ds.filter(lambda imgs, lbl: tf.equal(lbl, 0))
    bag_ds = tf.data.experimental.sample_from_datasets(
        [tumor_ds, normal_ds],
        weights=list(oversample_weight)
    )
    print(type(bag_ds))
    del parsed_ds, keyed_ds, tumor_ds, normal_ds
    gc.collect()
    return bag_ds

## Definition of MIL model

In [ ]:
from tensorflow.keras.utils import register_keras_serializable

class MILAttentionLayer(layers.Layer):
    """Implementation of the attention-based Deep MIL layer.

    Args:
      weight_params_dim: Positive Integer. Dimension of the weight matrix.
      kernel_initializer: Initializer for the `kernel` matrix.
      kernel_regularizer: Regularizer function applied to the `kernel` matrix.
      use_gated: Boolean, whether or not to use the gated mechanism.

    Returns:
      List of 2D tensors with BAG_SIZE length.
      The tensors are the attention scores after softmax with shape `(batch_size, 1)`.
    """

    def __init__(
        self,
        weight_params_dim,
        kernel_initializer="glorot_uniform",
        kernel_regularizer=None,
        use_gated=False,
        **kwargs,
    ):
        super().__init__(**kwargs)

        self.weight_params_dim = weight_params_dim
        self.use_gated = use_gated

        self.kernel_initializer = keras.initializers.get(kernel_initializer)
        self.kernel_regularizer = keras.regularizers.get(kernel_regularizer)

        self.v_init = self.kernel_initializer
        self.w_init = self.kernel_initializer
        self.u_init = self.kernel_initializer

        self.v_regularizer = self.kernel_regularizer
        self.w_regularizer = self.kernel_regularizer
        self.u_regularizer = self.kernel_regularizer

    def build(self, input_shape):
        # Input shape.
        # List of 2D tensors with shape: (batch_size, input_dim).
        input_dim = input_shape[0][1]

        self.v_weight_params = self.add_weight(
            shape=(input_dim, self.weight_params_dim),
            initializer=self.v_init,
            name="v",
            regularizer=self.v_regularizer,
            trainable=True,
        )

        self.w_weight_params = self.add_weight(
            shape=(self.weight_params_dim, 1),
            initializer=self.w_init,
            name="w",
            regularizer=self.w_regularizer,
            trainable=True,
        )

        if self.use_gated:
            self.u_weight_params = self.add_weight(
                shape=(input_dim, self.weight_params_dim),
                initializer=self.u_init,
                name="u",
                regularizer=self.u_regularizer,
                trainable=True,
            )
        else:
            self.u_weight_params = None

        self.input_built = True

    def compute_attention_scores(self, instance):
        # Reserve in-case "gated mechanism" used.
        original_instance = instance

        # tanh(v*h_k^T)
        instance = ops.tanh(ops.tensordot(instance, self.v_weight_params, axes=1))

        # for learning non-linear relations efficiently.
        if self.use_gated:
            instance = instance * ops.sigmoid(
                ops.tensordot(original_instance, self.u_weight_params, axes=1)
            )

        # w^T*(tanh(v*h_k^T)) / w^T*(tanh(v*h_k^T)*sigmoid(u*h_k^T))
        return ops.tensordot(instance, self.w_weight_params, axes=1)

    def call(self, inputs):
        # Assigning variables from the number of inputs.
        instances = [self.compute_attention_scores(instance) for instance in inputs]

        # Stack instances into a single tensor.
        instances = ops.stack(instances)

        # Apply softmax over instances such that the output summation is equal to 1.
        alpha = ops.softmax(instances, axis=0)

        # Split to recreate the same array of tensors we had as inputs.
        return [alpha[i] for i in range(alpha.shape[0])]

    def get_config(self):
            config = super(MILAttentionLayer, self).get_config()
            config.update({
                "weight_params_dim": self.weight_params_dim,
                "kernel_regularizer": keras.regularizers.serialize(self.kernel_regularizer),
                "use_gated": self.use_gated
            })
            return config

@register_keras_serializable(package="Custom")
class UnstackInstancesLayer(layers.Layer):
    def __init__(self, num, axis, **kwargs):
        super().__init__(**kwargs)
        self.num = num
        self.axis = axis

    def call(self, inputs):
        # returns a list of tensors: length=num, each [batch, 64]
        return tf.unstack(inputs, num=self.num, axis=self.axis)

    def get_config(self):
        config = super().get_config()
        config.update({'num': self.num, 'axis': self.axis})
        return config

@register_keras_serializable(package="Custom")
class ReshapeFeaturesLayer(tf.keras.layers.Layer):
    def __init__(self, bag_size, **kwargs):
        super().__init__(**kwargs)
        self.bag_size = bag_size

    def call(self, x):
        return tf.reshape(x, (-1, self.bag_size, tf.shape(x)[-1]))

@register_keras_serializable(package="Custom")
# Custom layer to extract the CLS token embedding
class ClassTokenExtractor(layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def call(self, inputs):
        # inputs shape: [batch*bag_size, num_tokens, hidden_dim]
        return inputs[:, 0, :]

    def get_config(self):
        return super().get_config()

@register_keras_serializable(package="Custom")
def Attention_weighted_sum(inputs):
    alpha, features = inputs
    expanded_alpha = tf.expand_dims(alpha, -1)  # shape: [batch, bag_size, 1]
    weighted_sum = tf.reduce_sum(expanded_alpha * features, axis=1)  # shape: [batch, feature_dim]
    return weighted_sum

@register_keras_serializable(package="Custom")
def flatten_bag_tensor(x):
    shape = tf.shape(x)  # dynamic: (batch, bag_size, H, W, C)
    #print(shape)
    return tf.reshape(x, (shape[0]*shape[1], patch_size, patch_size, 3))  # static known

@register_keras_serializable(package="Custom")
def flatten_output_shape(in_shape):
    # in_shape: (B, N, H, W, C) -> (None, H, W, C)  (B*N is dynamic)
    return (None, in_shape[2], in_shape[3], in_shape[4])

## Definition of Inference

In [ ]:
import time
import IPython
import threading

from sklearn.metrics import precision_score, recall_score, f1_score, \
accuracy_score, roc_auc_score, confusion_matrix, classification_report
from sklearn.metrics import confusion_matrix

def predict_model(test_dataset, model, threshold=0.5):

    # Evaluate
    tf.keras.backend.clear_session()
    strategy = tf.distribute.MirroredStrategy()

    with strategy.scope():
        num_test_steps = (val_bag_count * 2 // batch_size)
        print("Evaluating the testset with the model")
        eval_dict = model.evaluate(test_dataset, steps=num_test_steps, return_dict=True, verbose=1)
        print(
            f"The average loss and accuracy are "
            f"{eval_dict['loss']:.4f} and {eval_dict['accuracy']:.4f} respectively."
        )

        # Collect all true labels
        y_true_batches = []
        for _, bag_label in test_dataset:
            # bag_label is a 1-D tensor of shape (batch_size,)
            y_true_batches.append(bag_label.numpy())

        # concatenate into a single vector
        y_test = np.concatenate(y_true_batches, axis=0)
        print(f"Collected {len(y_test)} true labels.")

        # Get predictions for the entire dataset
        start = time.time()
        print("Predicting the label of the testset with the model")
        predictions = model.predict(test_dataset, steps=num_test_steps, verbose=1)  # shape: (num_bags, 1)
        elapsed_time = time.time()-start

    no_images = len(y_test)
    inference_time = 1000*elapsed_time/no_images
    inference_speed = no_images/inference_time

    # Get attention layer output model
    intermediate_model = keras.Model(
        model.input,
        model.get_layer("alpha").output
    )

    #print out the prediction in a  line
    try:
        print(f"🔹 [{' '.join([f'{float(p.item()):.3f}' for p in predictions[:16]])}]")
    except:
        print("predictions print did not work.")

    y_pred = [1 if p > threshold else 0 for p in predictions.flatten()]
    y_pred = np.array(y_pred)

    print("\norig labels:", y_test[:32], y_test.shape)
    print("prediction :", y_pred[:32], y_pred.shape)

    # Calculate Precision, Recall, and F1-Score
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred)

    print(f"{'Accuracy'.ljust(15)}\t{accuracy:.4f}")
    print(f"{'Precision'.ljust(15)}\t{precision:.4f}")
    print(f"{'Recall'.ljust(15)}\t{recall:.4f}")
    print(f"{'F1-Score'.ljust(15)}\t{f1:.4f}")
    print(f"{'roc_auc'.ljust(15)}\t{roc_auc:.4f}")

    # Calculate multilabel confusion matrix classification report
    conf_matrix = confusion_matrix(y_test, y_pred)
    class_report = classification_report(y_test, y_pred)

    results = {'accuracy':accuracy, 'precision':precision, 'recall':recall, 'f1':f1, 'roc_auc':roc_auc, \
            'conf_matrix':conf_matrix, 'class_report':class_report, \
            'inference_time':inference_time, 'inference_speed':inference_speed}

   # Create intermediate model to get MIL attention layer weights.
    intermediate_model = keras.Model(model.input, model.get_layer("alpha").output)

    # Predict MIL attention layer weights.
    intermediate_predictions = intermediate_model.predict(test_dataset, steps=num_test_steps, verbose=1)
    attention_weights = np.squeeze(np.swapaxes(intermediate_predictions, 1, 0))

    return results, attention_weights

## "LOOPING" - Testing with different hypers
>* Loading the datasets
>* Iterate it with hypers

In [ ]:
import itertools

from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.metrics import BinaryAccuracy, Precision, Recall, AUC
from tensorflow.keras.models import load_model

def count_records(tfrecord_path):
    count = 0
    for _ in tf.data.TFRecordDataset(tfrecord_path):
        count += 1
    return count

def print_hyperparameters(params, end=False):
    print(" 🔹 Hyperparameters in Use:")
    for key, value in params.items():
        print(f"  - {key}: {value}")

# Store best hyperparameter configuration
best_score = 0.0
best_params = None

test_count = count_records(test_tfrecords_path)
print(f"Total number of files: {test_count} at {test_tfrecords_path}")

# Iterate over all hyperparameter combinations
for learning_rate, patches_per_bag, num_bucket, norm_ratio, batch_size, bag_counts_pair in itertools.product(
    learning_rates, patches_per_bags, num_buckets, norm_ratios, batch_sizes, bag_counts_pairs):

    # Set them equal
    bag_size = patches_per_bag

    hyperparams = {
        "bag_count": bag_count,
        "val_bag_count":val_bag_count,
        "learning_rate": learning_rate,
        "batch_size": batch_size,
        "patches_per_bag": patches_per_bag,
        "bag_size" : bag_size,
        "num_bucket" : num_bucket,
        "norm_ratio" : norm_ratio
    }
    #model_path = (f"{model_pre_path}_lr{learning_rate}_ppg{patches_per_bag}"
    #              f"_bs{batch_size}_nb{num_bucket}_nr{norm_ratio}_model_{pretrained_vit}.keras")
    model_path = model_pre_path

    compiled_model_path = f"/home/elicer/Vision/MIL/models/ATOMized_{model_name}_{data_name}_{mag_label}.rbln"

    #compiled_model_path = (f"{compiled_pre_path}_lr{learning_rate}_ppg{patches_per_bag}"
    #              f"_bs{batch_size}_nb{num_bucket}_nr{norm_ratio}_model_{pretrained_vit}.rbln")

    print("Model will be saved as", compiled_model_path)

    print(f"\n🔥{model_name} Training by Hyper-Wisdom Seekers")
    print_hyperparameters(hyperparams)

    # test data need to be loaded from test.tfrecords
    test_dsx = tf.data.TFRecordDataset(test_tfrecords_path, num_parallel_reads=4)
    test_ds = create_bag_dataset(
        test_dsx, patches_per_bag=patches_per_bag, num_buckets=num_bucket,
        oversample_weight=(1.0-norm_ratio, norm_ratio)
    ).take(val_bag_count).cache().batch(batch_size, drop_remainder=True).prefetch(tf.data.AUTOTUNE)

    # Testing the model
    # Upload the best model
    print(f"✅Evaluation starts with loading the model from : {model_path}")
    model=load_model(model_path,  custom_objects={
            "MILAttentionLayer": MILAttentionLayer,
            "UnstackInstancesLayer": UnstackInstancesLayer,
            "ReshapeFeaturesLayer": ReshapeFeaturesLayer,
            "Attention_weighted_sum": Attention_weighted_sum})

    #model.summary()
    # Do something about the GPU model like this:
    func = tf.function(lambda input_img : model(input_img))
    input_info = [('input_img', [None, patches_per_bag, 224, 224, 3], tf.float32)]

    try:
        # Compling the model
        print("✅Compling the model")
        compiled_model = rebel.compile_from_tf_function(func, input_info)

        # Save the model before anything happens to it
        #compiled_model.save(compiled_model_path)

        # Evaluate the complied model
        #test_outputs, avg_attns = predict_model(test_ds, compiled_model)

        # Extract confusion matrix safely
        #if test_outputs['conf_matrix'].shape == (2, 2):
        #    TN, FP, FN, TP = test_outputs['conf_matrix'].ravel()
        #else:
        #    TN, FP, FN, TP = (None, None, None, None)

        # Prepare performance data
        perform_data ={
            "machine_name":[machine_name],
            'model_name': [model_name],
            'bag_count': [bag_count],
            'val_bag_count': [val_bag_count],
            'learning_rate': [learning_rate],
            "patches_per_bag": [patches_per_bag],
            "bag_size" : [bag_size],
            "num_bucket" : [num_bucket],
            "norm_ratio" : [norm_ratio],
            'batch_size': [batch_size],
            'drop_out' : [drop_out],
            'accuracy': [round(test_outputs['accuracy'], 4)],
            'precision': [round(test_outputs['precision'], 4)],
            'recall': [round(test_outputs['recall'], 4)],
            'f1-score': [round(test_outputs['f1'], 4)],
            'roc_auc': [round(test_outputs['roc_auc'], 4)],
            'confusion_matrix_TN': [TN],
            'confusion_matrix_FP': [FP],
            'confusion_matrix_FN': [FN],
            'confusion_matrix_TP': [TP],
            'inference_time(msec/image)': [round(test_outputs['inference_time'], 4)],
            'inference_speed(images/sec)': [round(test_outputs['inference_speed'], 1)],
            'comments' : [comments]}
        # Create DataFrame
        perform_df = pd.DataFrame(perform_data)

        # Save results to CSV (append mode)
        perform_df.to_csv(csv_file_path, mode='a', header=not os.path.exists(csv_file_path), index=False)

        # Update best hyperparameter set based on the AUC
        #if test_outputs['roc_auc'] > best_score:
        #    best_score = test_outputs['roc_auc']
        #    best_params = {
        #        "bag_count": bag_count,
        #        "val_bag_count":val_bag_count,
        #        'learning_rate': learning_rate,
        #        'batch_size': batch_size,
        #        'norm_ratio' : norm_ratio}
        #    save_path = f"./models/{best_score:.4f}_{model_path.split('/')[-1]}" # Ensure correct path format
        #    model.save(save_path)

        #del history
        #del model, train_ds, val_ds, test_ds, train_dsx, test_dsx, val_dsx
        #del perform_df, perform_data, hyperparams, #test_outputs
        #if(not inference_only):
        #    del history
        #gc.collect()
    except tf.errors.ResourceExhaustedError as e:
        print("⚠️ OOM or resource exhaustion detected, skipping this run...")
        print("Exception:", e)
        continue
    except Exception as e:
        print("⚠️ Unexpected error occurred:", e)
        continue

    # Check if monitoring should be reinitiated.
    #start_memory_monitoring(interval=5)
    #IPython.display.clear_output(wait=True)
    #print("✅ All memory released. You may train again with a clean conscience!")

print(f"\nBest accuracy: {best_score:.4f}")
print("Best hyper seekers:", best_params)
#stop_memory_monitoring()
print("🌿 End of the training.") #def train

Total number of files: 191821 at ['/home/elicer/data/camelyon16/tfrecord_10x/test.tfrecord']
Model will be saved as /home/elicer/Vision/MIL/models/ATOMized_MIL_CNN_model_came16_10x

🔥MIL_CNN_model Training by Hyper-Wisdom Seekers
 🔹 Hyperparameters in Use:
  - bag_count: 9000
  - val_bag_count: 3000
  - learning_rate: 1e-05
  - batch_size: 8
  - patches_per_bag: 70
  - bag_size: 70
  - num_bucket: 128
  - norm_ratio: 0.95
In Create_bag_dataset, Normal: 47985 Tumor: 2015
<class 'tensorflow.python.data.ops.directed_interleave_op._DirectedInterleaveDataset'>
✅Evaluation starts with loading the model from : ./models/0.9831_Jk_ViT_back_MIL_came16_10x_lr1e-05_ppg70_bs8_nb128_nr0.95_model_vit_base_patch16_224_imagenet.keras


Model: "ViT_MIL"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ bag_input           │ (None, 70, 224,   │          0 │ -                 │
│ (InputLayer)        │ 224, 3)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_bag         │ (None, 224, 224,  │          0 │ bag_input[0][0]   │
│ (Lambda)            │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ vi_t_backbone       │ (None, 197, 768)  │ 85,798,656 │ flatten_bag[0][0] │
│ (ViTBackbone)       │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cls_token           │ (None, 768)       │          0 │ vi_t_backbone[0]… │
│ (ClassTokenExtract… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_features    │ (None, 70, 768)   │          0 │ cls_token[0][0]   │
│ (ReshapeFeaturesLa… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ unstack_instances   │ [(None, 768),     │          0 │ reshape_features… │
│ (UnstackInstancesL… │ (None, 768),      │            │                   │
│                     │ (None, 768),      │            │                   │
│                     │ (None, 768),      │            │                   │
│                     │ (None, 768),      │            │                   │
│                     │ (None, 768),      │            │                   │
│                     │ (None, 768),      │            │                   │
│                     │ (None, 768),      │            │                   │
│                     │ (None, 768),      │            │                   │
│                     │ (None, 768),      │            │                   │
│                     │ (None, 768),      │            │                   │
│                     │ (None, 768),      │            │                   │
│                     │ (None, 768),      │            │                   │
│                     │ (None, 768),      │            │                   │
│                     │ (None, 768),      │            │                   │
│                     │ (None, 768),      │            │                   │
│                     │ (None, 768),      │            │                   │
│                     │ (None, 768),      │            │                   │
│                     │ (None, 768),      │            │                   │
│                     │ (None, 768),      │            │                   │
│                     │ (None, 768),      │            │                   │
│                     │ (None, 768),      │            │                   │
│                     │ (None, 768),      │            │                   │
│                     │ (None, 768),      │            │                   │
│                     │ (None, 768),      │            │                   │
│                     │ (None, 768),      │            │                   │
│                     │ (None, 768),      │            │                   │
│                     │ (None, 768),      │            │                   │
│                     │ (None, 768),      │            │                   │
│                     │ (None, 768),      │            │                   │
│                     │ (None, 768),      │            │                   │
│                     │ (None, 768),      │            │                   │
│                     │ (None, 768),      │            │                   │
│                     │ (None, 768),      │            │                 

 Total params: 258,724,233 (986.95 MB)

 Trainable params: 86,241,409 (328.98 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 172,482,824 (657.97 MB)

✅Compling the model
⚠️ Unexpected error occurred: The following operators are not implemented: {'Erfc'}

Best accuracy: 0.0000
Best hyper seekers: None
🌿 End of the training.


In [ ]:
print(test_outputs['conf_matrix'])

In [ ]:
TN, FP, FN, TP = test_outputs['conf_matrix'].ravel()
print(f"    Neg    Pos")
print(f"Neg {TN}   {FP}")
print(f"Pos {FN}   {TP}")